# Day 3 — Patterns 9 to 15 (Line-by-Line Walkthrough)

These are my solutions from `main.py`, copied exactly as written there (they are methods of the `Solution` class, hence the indentation and `self`). To run one, use `main.py`:

```python
sol = Solution()
sol.pattern09(5)
```

Every dry run below uses `n = 5`. Quick reminders:

- `range(k)` gives `0, 1, ..., k-1`.
- `print(x, end="")` stays on the same line; a bare `print()` jumps to the next row.

## Pattern 9 — Diamond

**Idea:** a diamond is just Pattern 7 (pyramid) stacked on top of Pattern 8 (inverted pyramid). So: two separate outer loops, one after the other. First loop draws the top half, second draws the bottom half.

Expected output for `n = 5`:

```
    *
   ***
  *****
 *******
*********
*********
 *******
  *****
   ***
    *
```

Note the widest row (9 stars) appears **twice** — once as the top loop's last row, once as the bottom loop's first.

In [ ]:
    def pattern09(self,n):
        for i in range(n):

            for j in range(n-i-1):
                print(" ", end = "")
            for j in range(2*i+1):
                print("*", end = "")
            print()

        for i in range(n):
            for j in range(i):
                print(" ", end = "")
            for j in range(2*n - 2*i - 1):
                print("*", end = "")
            print()

### Line by line

**Top half** (`for i in range(n)`), rows `i = 0..4`:

- `for j in range(n-i-1)` — leading spaces. Row 0 gets 4, row 4 gets 0. Spaces push the stars towards the centre.
- `for j in range(2*i+1)` — stars. Odd counts: 1, 3, 5, 7, 9. Odd numbers keep the shape symmetric around one centre star.
- `print()` — finish the row.

**Bottom half** (second `for i in range(n)`) — a fresh loop, so `i` restarts at 0:

- `for j in range(i)` — spaces now *grow*: 0, 1, 2, 3, 4.
- `for j in range(2*n - 2*i - 1)` — stars now *shrink*: 9, 7, 5, 3, 1.

### Dry run (n = 5)

| half | i | spaces | stars | row |
|------|---|--------|-------|-----|
| top | 0 | 4 | 1 | `    *` |
| top | 1 | 3 | 3 | `   ***` |
| top | 2 | 2 | 5 | `  *****` |
| top | 3 | 1 | 7 | ` *******` |
| top | 4 | 0 | 9 | `*********` |
| bottom | 0 | 0 | 9 | `*********` |
| bottom | 1 | 1 | 7 | ` *******` |
| bottom | 2 | 2 | 5 | `  *****` |
| bottom | 3 | 3 | 3 | `   ***` |
| bottom | 4 | 4 | 1 | `    *` |

Both halves use the variable name `i`, but the second loop starts over from 0 — that's why its formulas are mirror images of the first loop's.

## Pattern 10 — Half diamond (grow, then shrink)

**Idea:** star counts go `1 2 3 4 5 4 3 2 1`. Instead of two loops, this uses **one** loop over `2n+1` rows and an `if` to decide the phase: still climbing (`i <= n`) or coming down (`i > n`).

Expected output for `n = 5` — note the first and last lines are **blank**, because those rows print 0 stars:

```

*
**
***
****
*****
****
***
**
*

```

In [ ]:
    def pattern10(self, n):
        for i in range(2*n+1):
            if i>n:
                for j in range(2*n-i):
                    print("*", end = "")
            else:
                for j in range(i):
                    print("*", end = "")
            print()

### Line by line

- `for i in range(2*n+1)` — with n=5 this is `range(11)`, so `i` runs 0 to 10. Eleven rows total (9 visible + 2 blank).
- `if i>n:` — the **split point**. For `i = 6..10` we are past the peak, so we shrink.
- Shrink branch: `for j in range(2*n-i)` — stars = `10 - i`. At i=6 that's 4 stars; at i=10 it's 0.
- `else:` — for `i = 0..5` we are still climbing.
- Climb branch: `for j in range(i)` — stars = `i`. At i=0 that's 0 stars; at i=5 it's 5 (the peak).
- `print()` — sits **outside** the if/else, so it runs for every `i`. Even when 0 stars were printed, the newline still happens — that's exactly where the blank first and last lines come from.

### Careful dry run (n = 5, so the check is `i > 5`, and the shrink branch prints `10 - i` stars)

| i | `i > n`? | branch | stars | row shown |
|---|----------|--------|-------|-----------|
| 0 | no | climb: `range(0)` | 0 | (blank line) |
| 1 | no | climb: `range(1)` | 1 | `*` |
| 2 | no | climb | 2 | `**` |
| 3 | no | climb | 3 | `***` |
| 4 | no | climb | 4 | `****` |
| 5 | **no** — `5 > 5` is False | climb | 5 | `*****` |
| 6 | yes | shrink: `range(10-6)` | 4 | `****` |
| 7 | yes | shrink | 3 | `***` |
| 8 | yes | shrink | 2 | `**` |
| 9 | yes | shrink | 1 | `*` |
| 10 | yes | shrink: `range(0)` | 0 | (blank line) |

Two things people miss here:

1. **i = 5 takes the else branch.** `5 > 5` is False, so the peak row belongs to the climbing phase and prints `i = 5` stars.
2. The peak prints only **once** (unlike Pattern 9's doubled middle) because this is a single loop — it passes through i=5 exactly one time.

If the blank lines ever bother me: looping `for i in range(1, 2*n)` would skip both zero-star rows.

## Pattern 11 — Binary triangle (1-0 alternation)

**Idea:** every row alternates 1 and 0. Two decisions to make:

1. **What does the row start with?** Even rows (i = 0, 2, 4) start with 1; odd rows start with 0.
2. **How to flip while printing?** `num = 1 - num`. If num is 1 it becomes `1-1 = 0`; if num is 0 it becomes `1-0 = 1`. One line, no `if` — a toggle trick worth memorising.

Expected output for `n = 5` (numbers separated by a space, so each line also ends with one trailing space):

```
1
0 1
1 0 1
0 1 0 1
1 0 1 0 1
```

In [ ]:
    def pattern11(self,n):
        for i in range(n):
            num = 0
            if i%2 == 0:
                num = 1
            for j in range(i+1):
                print(num, end = " ")
                num = 1- num
            print()

### Line by line

- `for i in range(n)` — rows 0 to 4; row `i` holds `i+1` numbers.
- `num = 0` — default starting digit for the row.
- `if i%2 == 0: num = 1` — `%` is the remainder operator, so `i % 2 == 0` means "i is even". Even rows overwrite the start to 1. Net effect: even row starts with 1, odd row starts with 0.
- `for j in range(i+1)` — print `i+1` digits in this row.
- `print(num, end = " ")` — print the current digit plus a space, staying on the same line.
- `num = 1- num` — flip for the *next* position. Note the order: **print first, then flip.**
- `print()` — finish the row.

### Careful dry run (n = 5)

Row `i = 0` — even, so num starts at 1:
- j=0: print `1`, flip -> num = 0. Row: `1`

Row `i = 1` — odd, num stays 0:
- j=0: print `0`, flip -> 1
- j=1: print `1`, flip -> 0. Row: `0 1`

Row `i = 2` — even, num starts at 1:
- j=0: print `1`, flip -> 0
- j=1: print `0`, flip -> 1
- j=2: print `1`, flip -> 0. Row: `1 0 1`

Row `i = 3` — odd, num starts at 0:
- j=0: `0` -> 1
- j=1: `1` -> 0
- j=2: `0` -> 1
- j=3: `1` -> 0. Row: `0 1 0 1`

Row `i = 4` — even, num starts at 1: prints `1 0 1 0 1`.

Key detail: `num` is re-decided at the **top of every row**, so whatever the previous row left behind is thrown away. If the starting logic were removed and `num` lived outside the outer loop, the alternation would *continue* across rows instead of restarting — and the checkerboard look would break.

## Pattern 12 — Number crown (mirror with a gap)

**Idea:** each row has three jobs, in order: count **up** on the left, print a shrinking **gap** of spaces in the middle, count **down** on the right. Row by row the numbers grow and the gap shrinks, until the two sides touch.

Expected output for `n = 5`:

```
1        1
12      21
123    321
1234  4321
1234554321
```

Every row is exactly `2n = 10` characters wide: `(i+1)` digits + `2*(n-i-1)` spaces + `(i+1)` digits.

In [ ]:
    def pattern12(self,n):
        for i in range(n):
            for j in range(1,i+2):
                print(j, end = "")

            for j in range(2*(n-i-1)):
                print(" ", end = "")


            for j in range(i+1,0,-1):
                print(j, end = "")
                
            print()

### Line by line

- `for i in range(n)` — rows 0 to 4.
- `for j in range(1,i+2): print(j, end = "")` — left side, counting up. `range(1, i+2)` gives `1..i+1`. Row 0: just `1`. Row 3: `1234`.
- `for j in range(2*(n-i-1)): print(" ", end = "")` — the gap. Row 0: `2*(5-0-1) = 8` spaces. Every row down, the gap loses 2 (one seat taken on each side). Row 4: 0 spaces — the sides meet.
- `for j in range(i+1,0,-1): print(j, end = "")` — right side, counting **down**: `i+1, i, ..., 1`. The `-1` step makes `range` walk backwards; the stop value `0` is not included. Row 3: `4321`.
- `print()` — finish the row.

### Dry run (n = 5)

| i | left `1..i+1` | spaces `2*(n-i-1)` | right `i+1..1` | row |
|---|---------------|--------------------|----------------|-----|
| 0 | `1` | 8 | `1` | `1        1` |
| 1 | `12` | 6 | `21` | `12      21` |
| 2 | `123` | 4 | `321` | `123    321` |
| 3 | `1234` | 2 | `4321` | `1234  4321` |
| 4 | `12345` | 0 | `54321` | `1234554321` |

Why the width never changes: `2*(i+1) + 2*(n-i-1) = 2n` — the `i` terms cancel out. That's the logic hidden inside the space formula.

## Pattern 13 — Counting triangle

**Idea:** one single counter that **never resets**. It starts at 1 before any row begins and keeps incrementing across row boundaries — like token numbers at a bank counter: the next person continues from where the last left off, no restart per row.

Expected output for `n = 5` (a space after each number):

```
1
2 3
4 5 6
7 8 9 10
11 12 13 14 15
```

In [ ]:
    def pattern13(self,n):
        num = 1
        for i in range(n):
            for j in range(1,i+2):
                print(num ,end = " ")
                num += 1

            print()

### Line by line

- `num = 1` — declared **before** the outer loop. This is the whole trick: `num` lives across all rows.
- `for i in range(n)` — rows 0 to 4; row `i` gets `i+1` numbers.
- `for j in range(1,i+2)` — runs `i+1` times. Note `j` itself is never printed — it only counts how many slots this row has.
- `print(num ,end = " ")` — print the running counter.
- `num += 1` — step to the next number. Because `num` is outside the loops, this carries into the next row.
- `print()` — finish the row.

### Dry run (n = 5)

- Row 0: 1 slot -> `1` (num is now 2)
- Row 1: 2 slots -> `2 3` (num is now 4)
- Row 2: 3 slots -> `4 5 6` (num is now 7)
- Row 3: 4 slots -> `7 8 9 10` (num is now 11)
- Row 4: 5 slots -> `11 12 13 14 15` (num ends at 16)

Total printed: 1+2+3+4+5 = 15 = n(n+1)/2. Compare with Pattern 14 next — same skeleton, but the per-row variable is reset *inside* the loop. Where you initialise decides whether a counter continues or restarts.

## Pattern 14 — Alphabet triangle (A, AB, ABC, ...)

**Idea:** computers store characters as numbers, called **ASCII** codes. `'A'` is 65, `'B'` is 66, and so on. `chr(number)` converts a code back into its character: `chr(65)` gives `'A'`. So "print the alphabet" becomes "count from 65 and convert". Since every row restarts from `'A'`, the counter is reset **inside** the outer loop.

Expected output for `n = 5`:

```
A
AB
ABC
ABCD
ABCDE
```

In [ ]:
    def pattern14(self,n):
        for i in range(n):
            char = 65
            for j in range(1,i+2):
                print(chr(char) ,end = "")
                char+= 1
            print()

### Line by line

- `for i in range(n)` — rows 0 to 4.
- `char = 65` — reset to `'A'` at the **start of every row**. The position of this line matters more than anything else here.
- `for j in range(1,i+2)` — `i+1` letters in row `i`.
- `print(chr(char) ,end = "")` — convert the code to its letter, print without a newline.
- `char+= 1` — 65 -> 66 -> 67, meaning A -> B -> C, within this row only.
- `print()` — finish the row.

### Dry run (n = 5)

- Row 0: char 65 -> `A`
- Row 1: 65, 66 -> `AB`
- Row 2: 65, 66, 67 -> `ABC`
- Row 3: -> `ABCD`
- Row 4: -> `ABCDE`

If `char = 65` moved *above* the outer loop, the output would become `A / BC / DEF / GHIJ / KLMNO` — a continuous alphabet, exactly like Pattern 13's continuous numbers.

Bonus: `ord()` is the reverse of `chr()` — `ord('A')` gives 65. If I forget the code, I can just write `char = ord('A')`.

## Pattern 15 — Reverse alphabet triangle

**Idea:** identical machinery to Pattern 14 — reset `char = 65` per row, count up within the row. Only one thing changes: the row **length** shrinks. Row `i` prints `n - i` letters instead of `i + 1`.

Expected output for `n = 5`:

```
ABCDE
ABCD
ABC
AB
A
```

In [ ]:
    def pattern15(self,n):
        for i in range(n):
            char = 65
            for j in range(n-i):
                print(chr(char), end = "")
                char += 1
            print()

### Line by line

- `for i in range(n)` — rows 0 to 4.
- `char = 65` — every row starts from `'A'` again.
- `for j in range(n-i)` — the shrink: row 0 gets 5 letters, row 1 gets 4, ..., row 4 gets 1.
- `print(chr(char), end = "")` then `char += 1` — print the current letter, step to the next.
- `print()` — finish the row.

### Dry run (n = 5)

| i | letters `n-i` | row |
|---|---------------|-----|
| 0 | 5 | `ABCDE` |
| 1 | 4 | `ABCD` |
| 2 | 3 | `ABC` |
| 3 | 2 | `AB` |
| 4 | 1 | `A` |

Patterns 13, 14 and 15 are really one lesson: the outer loop controls **how many**, the inner variable controls **what**, and *where that variable is initialised* (inside vs outside the outer loop) controls whether it **resets or continues**.

## Pending — to solve next in main.py

Not written yet. Hints only, so the solving stays with me.

### 1. Prime check
- A prime has exactly two divisors: 1 and itself. Handle `n < 2` first — 0 and 1 are **not** prime.
- Don't loop all the way to `n`. Divisors come in pairs, and one of each pair always sits at or below the square root of `n`. So loop only `while i * i <= n`.
- The moment `n % i == 0`, stop — not prime.
- Test with: 1 (no), 2 (yes), 9 (no — this one catches a wrong loop boundary), 25 (no), 29 (yes).

### 2. GCD — Euclid's algorithm
- Core identity: `gcd(a, b) == gcd(b, a % b)`. Keep replacing until the second number becomes 0; the first number is then the answer.
- Loop shape: `while b != 0:` swap `a, b = b, a % b`, then return `a`.
- Trace (48, 18) by hand before coding: (48, 18) -> (18, 12) -> (12, 6) -> (6, 0) -> answer 6.
- Edge check: what if the smaller number comes first? Try (18, 48) — the first step swaps them automatically.

### 3. LCM
- No looping needed. Use the identity `a * b == gcd(a, b) * lcm(a, b)`.
- So `lcm = (a * b) // gcd(a, b)` — reuse the gcd function just written. Use `//` (integer division), not `/`.
- Test with (4, 6) -> 12, and (7, 5) -> 35 (co-prime pair: gcd is 1, so lcm is just the product).

Today's function lesson applies directly here: all three should **return** their answers, not print them — that's the only way `lcm` can call `gcd` and actually use the value.